In [0]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from pyspark.sql import functions as F

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

In [0]:

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"
DATA_SEPARACAO = "2025-12-31"
N_DIAS_ALVO = 30

fato_titulos = spark.table(f"{GOLD}.fato_titulos")

base = (
    fato_titulos
    .withColumn("dias_desde_vencimento",
                F.datediff(F.lit(DATA_CORTE).cast("date"), F.col("data_vencimento")))
    .withColumn("entra_treino",
                F.col("dias_desde_vencimento") > N_DIAS_ALVO)
    .withColumn("alvo",
                F.when(~F.col("entra_treino"), F.lit(None))
                 .when(F.col("dias_atraso") > N_DIAS_ALVO, F.lit(1))
                 .when(F.col("dias_atraso") <= N_DIAS_ALVO, F.lit(0)))
)

In [0]:
por_mes = (
    base
    .filter(F.col("entra_treino"))
    .withColumn("mes_emissao", F.date_format("data_base", "yyyy-MM"))
    .groupBy("mes_emissao")
    .agg(
        F.count(F.lit(1)).alias("qtd_titulos"),
        F.sum("alvo").alias("qtd_alvo_1"),
    )
    .withColumn("taxa_alvo",
                F.round(F.col("qtd_alvo_1") / F.col("qtd_titulos") * 100, 2))
    .orderBy("mes_emissao")
)

por_mes.show(30)

In [0]:
base_modelo = (
    base
    .filter(F.col("entra_treino"))
    .withColumn("conjunto",
                F.when(F.col("data_base") <= F.lit(DATA_SEPARACAO).cast("date"), F.lit("treino"))
                 .when(F.col("data_base") > F.lit(DATA_SEPARACAO).cast("date"), F.lit("teste")))
)

(
    base_modelo
    .groupBy("conjunto")
    .agg(
        F.count(F.lit(1)).alias("qtd_titulos"),
        F.sum("alvo").alias("qtd_alvo_1"),
    )
    .withColumn("taxa_alvo",
                F.round(F.col("qtd_alvo_1") / F.col("qtd_titulos") * 100, 2))
    .show()
)



In [0]:
# Passo 1: a dimensão de clientes, só com o que o modelo usa
clientes_dim = (
    spark.table(f"{SILVER}.clientes")
    .select("cod_cliente", "ramo_atividade", "uf")
)

# Passo 2: juntar com os títulos (títulos à esquerda: o grão é o título)
dados_modelo = base_modelo.join(clientes_dim, on="cod_cliente", how="left")

# Passo 2b: juntar as features de histórico do cliente (point-in-time)
features_hist = spark.table(f"{GOLD}.features_historico_cliente")
dados_modelo = dados_modelo.join(features_hist, on="cod_fatura", how="left")

# Passo 3: criar a feature do mês
dados_modelo = dados_modelo.withColumn("mes_vencimento", F.month("data_vencimento"))

# Passo 4: ficar só com as colunas do modelo (protege contra vazamento)
dados_modelo = dados_modelo.select(
    "cod_cliente", "cod_fatura", # chave
    "valor", "prazo_dias", "mes_vencimento", "ramo_atividade", "uf", # features
    "alvo", "conjunto", # alvo e separação
    "atraso_medio_hist", "qtd_titulos_hist" # features de histórico
)

# Validação
print(dados_modelo.count())
dados_modelo.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dados_modelo.columns]
).show()

In [0]:
FEATURES_NUM = ["valor"]
FEATURES_CAT = ["ramo_atividade", "uf", "mes_vencimento", "prazo_dias"]

dados_pd = dados_modelo.toPandas()
dados_pd["valor"] = dados_pd["valor"].astype(float)

treino = dados_pd[dados_pd["conjunto"] == "treino"]
teste = dados_pd[dados_pd["conjunto"] == "teste"]

X_treino = treino[FEATURES_NUM + FEATURES_CAT]
y_treino = treino["alvo"]
X_teste = teste[FEATURES_NUM + FEATURES_CAT]
y_teste = teste["alvo"]

print(X_treino.shape, y_treino.sum())
print(X_teste.shape, y_teste.sum())

In [0]:
preparacao = ColumnTransformer([
    ("numericas", StandardScaler(), FEATURES_NUM),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

pipeline = Pipeline([
    ("preparacao", preparacao),
    ("modelo", LogisticRegression(max_iter=1000)),
])

pipeline.fit(X_treino, y_treino)
prob = pipeline.predict_proba(X_treino)[:, 1]
print(prob.mean())

In [0]:
def captura_no_topo(y, prob, fracao=0.10):
    tabela = pd.DataFrame({"y": y.values, "prob": prob})        # junta alvo e risco
    tabela = tabela.sort_values("prob", ascending=False)        # maior risco primeiro
    n_topo = int(len(tabela) * fracao)                          # 10% de 6.669 = 666 títulos
    return tabela["y"].head(n_topo).sum() / tabela["y"].sum()   # alvos 1 no topo / todos os alvos 1

prob_teste = pipeline.predict_proba(X_teste)[:, 1]

print(f"Captura no topo 10%: {captura_no_topo(y_teste, prob_teste):.1%}")
print(f"ROC-AUC: {roc_auc_score(y_teste, prob_teste):.3f}")
print(f"PR-AUC: {average_precision_score(y_teste, prob_teste):.3f}")

In [0]:
mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(run_name="baseline_regressao_logistica"):
    # O treino acontece dentro da run, para o modelo ficar ligado aos parâmetros
    pipeline.fit(X_treino, y_treino)
    prob_teste = pipeline.predict_proba(X_teste)[:, 1]

    # Parâmetros: o "como fiz"
    mlflow.log_param("algoritmo", "LogisticRegression")
    mlflow.log_param("class_weight", "nenhum")
    mlflow.log_param("features_num", FEATURES_NUM)
    mlflow.log_param("features_cat", FEATURES_CAT)
    mlflow.log_param("n_dias_alvo", N_DIAS_ALVO)
    mlflow.log_param("data_separacao", DATA_SEPARACAO)
    mlflow.log_param("data_corte", DATA_CORTE)

    # Métricas: o "quanto deu", sempre no teste
    mlflow.log_metric("roc_auc_teste", roc_auc_score(y_teste, prob_teste))
    mlflow.log_metric("pr_auc_teste", average_precision_score(y_teste, prob_teste))
    mlflow.log_metric("captura_top10_teste", captura_no_topo(y_teste, prob_teste))

    # Artefato: o modelo treinado, com um exemplo de entrada
    mlflow.sklearn.log_model(pipeline, name="modelo", input_example=X_teste.head(5))

In [0]:
# Converter para pandas (o scikit-learn trabalha com pandas)
dados_modelo_pd = dados_modelo.toPandas()
dados_modelo_pd["valor"] = dados_modelo_pd["valor"].astype(float)   # decimal do Spark vira float

# Flag de cliente sem histórico conhecido na emissão
FEATURES_HIST = ["atraso_medio_hist", "sem_historico"]
dados_modelo_pd["sem_historico"] = dados_modelo_pd["atraso_medio_hist"].isna().astype(int)

# Separar treino e teste
treino = dados_modelo_pd[dados_modelo_pd["conjunto"] == "treino"]
teste = dados_modelo_pd[dados_modelo_pd["conjunto"] == "teste"]

# Features e alvo
X_treino_h = treino[FEATURES_NUM + FEATURES_HIST + FEATURES_CAT]
X_teste_h = teste[FEATURES_NUM + FEATURES_HIST + FEATURES_CAT]
y_treino = treino["alvo"].astype(int)
y_teste = teste["alvo"].astype(int)

# Validação
print(X_treino_h.shape, X_teste_h.shape)        # esperado (18012, 7) e (6669, 7)
print(y_treino.sum(), y_teste.sum())            # esperado 1144 e 494
print(X_treino_h["sem_historico"].sum(), X_teste_h["sem_historico"].sum())

In [0]:
def captura_top(y, score, frac=0.10):
    y = np.asarray(y)
    k = int(np.ceil(len(y) * frac))
    topo = np.argsort(-np.asarray(score))[:k]
    return y[topo].sum() / y.sum()

def avaliar(y, score):
    return {
        "pr_auc_teste": average_precision_score(y, score),
        "captura_top10_teste": captura_top(y, score),
        "roc_auc_teste": roc_auc_score(y, score),
    }

In [0]:
mediana_treino = treino["atraso_medio_hist"].median()
score_regra = teste["atraso_medio_hist"].fillna(mediana_treino)

metricas_regra = avaliar(y_teste, score_regra)
print(metricas_regra)

with mlflow.start_run(run_name="regra_atraso_medio_hist", description="Sem modelo: ordena os títulos pelo atraso médio conhecido do cliente na emissão."):
    mlflow.log_params({"algoritmo": "regra_simples", "ordenacao": "atraso_medio_hist", "nulos": "mediana_treino"})
    mlflow.log_metrics(metricas_regra)

In [0]:
preparo_h = ColumnTransformer([
    ("num", StandardScaler(), FEATURES_NUM),
    ("hist", Pipeline([
        ("imputa", SimpleImputer(strategy="median")),
        ("escala", StandardScaler()),
    ]), ["atraso_medio_hist"]),
    ("flag", "passthrough", ["sem_historico"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

modelo_h = Pipeline([
    ("preparo", preparo_h),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo_h.fit(X_treino_h, y_treino)
prob_teste_h = modelo_h.predict_proba(X_teste_h)[:, 1]

metricas_h = avaliar(y_teste, prob_teste_h)
print(metricas_h)
print("média prevista no treino:", modelo_h.predict_proba(X_treino_h)[:, 1].mean())   # perto de 0,0635

In [0]:
with mlflow.start_run(run_name="regressao_logistica_historico", description="Baseline + atraso médio point-in-time do cliente e flag sem_historico."):
    mlflow.log_params({
        "algoritmo": "regressao_logistica",
        "class_weight": None,
        "features_num": FEATURES_NUM,
        "features_hist": FEATURES_HIST,
        "features_cat": FEATURES_CAT,
        "nulos_hist": "mediana_treino + flag",
        "n_dias_alvo": N_DIAS_ALVO,
        "data_separacao": DATA_SEPARACAO,
        "data_corte": DATA_CORTE,
    })
    mlflow.log_metrics(metricas_h)
    mlflow.sklearn.log_model(modelo_h, name="modelo", input_example=X_treino_h.head(5))

In [0]:
# Probabilidades do baseline recalculadas no mesmo teste das outras linhas
X_teste_base = teste[FEATURES_NUM + FEATURES_CAT]
prob_teste_baseline = pipeline.predict_proba(X_teste_base)[:, 1]

metricas_baseline = avaliar(y_teste, prob_teste_baseline)

pd.DataFrame(
    [metricas_baseline, metricas_regra, metricas_h],
    index=["baseline", "regra_simples", "modelo_historico"],
).round(3)

In [0]:
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")

NOME_MODELO = f"{GOLD}.modelo_risco_atraso"
ID_MODELO_H = "m-5526faeefd3c4e71b2f7c4a82ca1b3fa"   # run regressao_logistica_historico

# Registrar a versão no Unity Catalog
versao = mlflow.register_model(f"models:/{ID_MODELO_H}", NOME_MODELO)

# Apelido "campeao": a pontuação carrega sempre o modelo com esse apelido, sem fixar número de versão
MlflowClient().set_registered_model_alias(NOME_MODELO, "campeao", versao.version)

# Validação: o modelo carregado do catálogo prevê igual ao da memória
modelo_carregado = mlflow.sklearn.load_model(f"models:/{NOME_MODELO}@campeao")
prob_carregado = modelo_carregado.predict_proba(X_teste_h)[:, 1]

print("versão:", versao.version)                                 # esperado 1
print("previsões iguais:", np.allclose(prob_carregado, prob_teste_h))  # esperado True

In [0]:
CAMINHO_GABARITO = "/Volumes/workspace/s4lake_bronze/landing/_gabarito/perfil_clientes.csv"

gabarito_pd = pd.read_csv(CAMINHO_GABARITO, dtype=str)

print(gabarito_pd.shape)
print(gabarito_pd.columns.tolist())
print(gabarito_pd.head())

In [0]:
# O gabarito só valida: nunca entra como feature (seria vazamento)
ORDEM_SEGMENTOS = ["pontual", "toleravel", "atrasador", "critico"]

print(gabarito_pd["segmento_pagamento"].value_counts())   # esperado perto de 440 / 240 / 96 / 24

# Títulos do teste com a probabilidade do modelo e o alvo real
teste_aval = teste[["cod_cliente", "alvo"]].copy()
teste_aval["prob"] = prob_teste_h          # mesma ordem: prob_teste_h veio de teste

# Juntar o segmento oculto de cada cliente
teste_aval = teste_aval.merge(
    gabarito_pd[["KUNNR", "segmento_pagamento"]],
    left_on="cod_cliente", right_on="KUNNR", how="left",
)
print("títulos sem segmento:", teste_aval["segmento_pagamento"].isna().sum())   # esperado 0

# Risco previsto x taxa real, por segmento
por_segmento = (
    teste_aval
    .groupby("segmento_pagamento")
    .agg(qtd_titulos=("alvo", "size"), prob_media=("prob", "mean"), taxa_real=("alvo", "mean"))
    .reindex(ORDEM_SEGMENTOS)
    .round(3)
)
print(por_segmento)

# O cliente 178 no gabarito
print(gabarito_pd[gabarito_pd["KUNNR"] == "0000100178"])